# I04 - Advanced CNN Architectures

**Interactive Application:** [CNN Explorer](https://nexageapps.github.io/AI/compsci714/week3/cnn-explorer/) - Explore CNN architectures interactively, from LeNet to VGG, and build your own in the Playground!

**Level:** Intermediate

---

## Learning Objectives

By the end of this lesson, you will:
1. Understand ResNet and skip connections
2. Learn VGG architecture principles
3. Implement Inception modules
4. Explore EfficientNet and compound scaling
5. Compare different CNN architectures

---

## Prerequisites

- Completed B09 (CNNs) and I01-I03
- Understanding of convolutional layers
- Familiarity with batch normalization

---

In [ ]:
# Import required libraries
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print(f"TensorFlow version: {tf.__version__}")

## 1. The Evolution of CNN Architectures

### Historical Timeline

1. **LeNet-5 (1998)**: First successful CNN
2. **AlexNet (2012)**: ImageNet breakthrough
3. **VGG (2014)**: Deeper with small filters
4. **GoogLeNet/Inception (2014)**: Multi-scale features
5. **ResNet (2015)**: Skip connections, very deep
6. **EfficientNet (2019)**: Compound scaling

### Key Innovations

- **Depth**: Going deeper improves performance
- **Skip connections**: Enable training very deep networks
- **Multi-scale**: Process features at different scales
- **Efficiency**: Balance accuracy and computational cost

In [ ]:
# Load CIFAR-10 dataset
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

# Normalize
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# Use subset for faster training
x_train_small = x_train[:10000]
y_train_small = y_train[:10000]

print(f"Training samples: {x_train_small.shape}")
print(f"Test samples: {x_test.shape}")

## 2. ResNet: Residual Networks

### The Vanishing Gradient Problem

**Problem:** As networks get deeper, gradients become very small (vanish) or very large (explode)

**Solution:** Skip connections (residual connections)

### Residual Block

Instead of learning $H(x)$, learn the residual $F(x) = H(x) - x$:

$$H(x) = F(x) + x$$

**Benefits:**
- Easier to optimize
- Enables training 100+ layer networks
- Better gradient flow
- Identity mapping as fallback

In [ ]:
def residual_block(x, filters, kernel_size=3, stride=1, conv_shortcut=False):
    """ResNet residual block
    
    Args:
        x: Input tensor
        filters: Number of filters
        kernel_size: Convolution kernel size
        stride: Stride for first convolution
        conv_shortcut: Use convolution for shortcut (when dimensions change)
    """
    # Shortcut path
    if conv_shortcut:
        shortcut = layers.Conv2D(filters, 1, strides=stride, padding='same')(x)
        shortcut = layers.BatchNormalization()(shortcut)
    else:
        shortcut = x
    
    # Main path
    x = layers.Conv2D(filters, kernel_size, strides=stride, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    
    x = layers.Conv2D(filters, kernel_size, padding='same')(x)
    x = layers.BatchNormalization()(x)
    
    # Add shortcut
    x = layers.Add()([x, shortcut])
    x = layers.Activation('relu')(x)
    
    return x

def create_resnet(input_shape=(32, 32, 3), num_classes=10):
    """Create a ResNet-style model"""
    inputs = keras.Input(shape=input_shape)
    
    # Initial convolution
    x = layers.Conv2D(64, 3, padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    
    # Residual blocks
    x = residual_block(x, 64)
    x = residual_block(x, 64)
    
    x = residual_block(x, 128, stride=2, conv_shortcut=True)
    x = residual_block(x, 128)
    
    x = residual_block(x, 256, stride=2, conv_shortcut=True)
    x = residual_block(x, 256)
    
    # Global pooling and classification
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(512, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    model = Model(inputs, outputs, name='ResNet')
    return model

resnet_model = create_resnet()
print("ResNet model created")
resnet_model.summary()

## 3. VGG: Very Deep Networks

### VGG Architecture Principles

**Key ideas:**
1. Use only 3×3 convolutions
2. Stack multiple conv layers before pooling
3. Double filters after each pooling
4. Very deep (16-19 layers)

**Why 3×3 filters?**
- Two 3×3 convs = one 5×5 receptive field
- Three 3×3 convs = one 7×7 receptive field
- Fewer parameters, more non-linearity

In [ ]:
def vgg_block(x, filters, num_convs):
    """VGG block: multiple 3x3 convolutions followed by max pooling"""
    for _ in range(num_convs):
        x = layers.Conv2D(filters, 3, padding='same', activation='relu')(x)
    x = layers.MaxPooling2D(2, strides=2)(x)
    return x

def create_vgg(input_shape=(32, 32, 3), num_classes=10):
    """Create a VGG-style model (simplified for CIFAR-10)"""
    inputs = keras.Input(shape=input_shape)
    
    # VGG blocks
    x = vgg_block(inputs, 64, 2)   # 32x32 -> 16x16
    x = vgg_block(x, 128, 2)        # 16x16 -> 8x8
    x = vgg_block(x, 256, 3)        # 8x8 -> 4x4
    x = vgg_block(x, 512, 3)        # 4x4 -> 2x2
    
    # Classifier
    x = layers.Flatten()(x)
    x = layers.Dense(4096, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(4096, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    model = Model(inputs, outputs, name='VGG')
    return model

vgg_model = create_vgg()
print("VGG model created")
print(f"Total parameters: {vgg_model.count_params():,}")

## 4. Inception: Multi-Scale Features

### Inception Module

**Problem:** What filter size should we use?

**Solution:** Use multiple filter sizes in parallel!

**Inception module includes:**
- 1×1 convolutions (dimensionality reduction)
- 3×3 convolutions
- 5×5 convolutions
- 3×3 max pooling
- Concatenate all outputs

**Benefits:**
- Captures features at multiple scales
- 1×1 convs reduce computational cost
- More efficient than VGG

In [ ]:
def inception_module(x, filters_1x1, filters_3x3_reduce, filters_3x3,
                    filters_5x5_reduce, filters_5x5, filters_pool_proj):
    """Inception module with multiple parallel paths"""
    
    # 1x1 convolution branch
    branch1 = layers.Conv2D(filters_1x1, 1, padding='same', activation='relu')(x)
    
    # 3x3 convolution branch
    branch2 = layers.Conv2D(filters_3x3_reduce, 1, padding='same', activation='relu')(x)
    branch2 = layers.Conv2D(filters_3x3, 3, padding='same', activation='relu')(branch2)
    
    # 5x5 convolution branch
    branch3 = layers.Conv2D(filters_5x5_reduce, 1, padding='same', activation='relu')(x)
    branch3 = layers.Conv2D(filters_5x5, 5, padding='same', activation='relu')(branch3)
    
    # Max pooling branch
    branch4 = layers.MaxPooling2D(3, strides=1, padding='same')(x)
    branch4 = layers.Conv2D(filters_pool_proj, 1, padding='same', activation='relu')(branch4)
    
    # Concatenate all branches
    output = layers.Concatenate()([branch1, branch2, branch3, branch4])
    
    return output

def create_inception(input_shape=(32, 32, 3), num_classes=10):
    """Create an Inception-style model"""
    inputs = keras.Input(shape=input_shape)
    
    # Initial convolutions
    x = layers.Conv2D(64, 3, padding='same', activation='relu')(inputs)
    x = layers.MaxPooling2D(2)(x)
    
    # Inception modules
    x = inception_module(x, 64, 96, 128, 16, 32, 32)
    x = inception_module(x, 128, 128, 192, 32, 96, 64)
    x = layers.MaxPooling2D(2)(x)
    
    x = inception_module(x, 192, 96, 208, 16, 48, 64)
    x = inception_module(x, 160, 112, 224, 24, 64, 64)
    
    # Global pooling and classification
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    model = Model(inputs, outputs, name='Inception')
    return model

inception_model = create_inception()
print("Inception model created")
print(f"Total parameters: {inception_model.count_params():,}")

## 5. EfficientNet: Compound Scaling

### The Scaling Problem

**Traditional approach:** Scale one dimension
- Depth (more layers)
- Width (more channels)
- Resolution (larger images)

**EfficientNet approach:** Scale all three together!

### Compound Scaling Formula

$$depth: d = \alpha^\phi$$
$$width: w = \beta^\phi$$
$$resolution: r = \gamma^\phi$$

Subject to: $\alpha \cdot \beta^2 \cdot \gamma^2 \approx 2$

**Benefits:**
- Better accuracy-efficiency trade-off
- Systematic scaling strategy
- State-of-the-art results

In [ ]:
# Compare model architectures
models = {
    'ResNet': create_resnet(),
    'VGG': create_vgg(),
    'Inception': create_inception()
}

# Compare model sizes
print("\nModel Comparison:")
print("=" * 60)
for name, model in models.items():
    params = model.count_params()
    print(f"{name:15} | Parameters: {params:,}")
print("=" * 60)

In [ ]:
# Train and compare models (using ResNet as example)
print("\nTraining ResNet model...")

resnet_model.compile(
    optimizer=keras.optimizers.Adam(0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history = resnet_model.fit(
    x_train_small, y_train_small,
    epochs=30,
    batch_size=128,
    validation_split=0.2,
    verbose=1
)

# Evaluate
test_loss, test_acc = resnet_model.evaluate(x_test, y_test, verbose=0)
print(f"\nTest Accuracy: {test_acc:.4f}")

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].plot(history.history['loss'], label='Training Loss', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('ResNet Training Loss', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['accuracy'], label='Training Accuracy', linewidth=2)
axes[1].plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Accuracy', fontsize=12)
axes[1].set_title('ResNet Training Accuracy', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Architecture Design Principles

### Key Lessons from Modern CNNs

1. **Depth matters** - Deeper networks learn better representations
2. **Skip connections** - Enable training very deep networks
3. **Batch normalization** - Stabilizes training
4. **Global average pooling** - Reduces parameters vs fully connected
5. **Multi-scale features** - Capture information at different scales
6. **Efficient design** - Balance accuracy and computational cost

### When to Use Each Architecture

| Architecture | Best For | Pros | Cons |
|-------------|----------|------|------|
| **ResNet** | General purpose | Easy to train, very deep | More parameters |
| **VGG** | Transfer learning | Simple, uniform | Very large model |
| **Inception** | Multi-scale tasks | Efficient, accurate | Complex architecture |
| **EfficientNet** | Mobile/edge | Best efficiency | Requires careful tuning |

## Summary

In this lesson, you learned:

1. ✅ ResNet and skip connections for very deep networks
2. ✅ VGG architecture with small filters
3. ✅ Inception modules for multi-scale features
4. ✅ EfficientNet and compound scaling
5. ✅ Architecture design principles

**Key Takeaways:**
- Skip connections enable training 100+ layer networks
- Multiple 3×3 convs are better than one large filter
- Multi-scale features improve performance
- Balance accuracy, speed, and model size
- Use pre-trained models when possible

**Next Steps:**
- Move to I05 - Transfer Learning and Fine-tuning
- Experiment with different architectures
- Try pre-trained models from TensorFlow/PyTorch

---

**References:**
- He et al. (2015): "Deep Residual Learning for Image Recognition"
- Simonyan & Zisserman (2014): "Very Deep Convolutional Networks"
- Szegedy et al. (2015): "Going Deeper with Convolutions"
- Tan & Le (2019): "EfficientNet: Rethinking Model Scaling"

---

## Learning Progress Tracker

Use this section to track your learning progress for this lesson.

### Completion Status
- [ ] Lesson completed
- [ ] All code cells executed successfully
- [ ] Understood all key concepts
- [ ] Completed practice exercises (if any)

### Dates
- **First Completed:** ____/____/____
- **Last Reviewed:** ____/____/____
- **Next Review:** ____/____/____ (Recommended: 1 week, 1 month, 3 months)

### Understanding Level
Rate your understanding (1-5): _____ / 5

- 1 = Need to review completely
- 2 = Understood basics, need more practice
- 3 = Good understanding, minor gaps
- 4 = Strong understanding, can explain to others
- 5 = Mastered, can apply in projects

### Notes & Reflections
```
Write your notes here:
- What concepts were challenging?
- What was interesting or surprising?
- How can you apply this in projects?
- Questions to explore further?




```

### Key Concepts to Remember (I04)
- [ ] ResNet architecture
- [ ] VGG networks
- [ ] Inception modules
- [ ] EfficientNet

---